# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tabassumrafiq/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule

I will prioritize content for review when it has meaningful observed search visibility
but a weaker average search position.

The baseline combines observed GSC impressions with average search position.
Higher impressions indicate measurable search visibility, while a weaker average
position increases review priority.

This is a directional decision-support rule, not proof that the content must be refreshed.

### Reason codes

- `VISIBLE_POSITION_REVIEW` — the content has observed search visibility and a weaker
  average search position, so it is prioritized for review.
- `NO_HIGH_PRIORITY_SIGNAL` — the content does not meet the high-priority score
  threshold under this baseline.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import duckdb

# Initialize an in-memory duckdb connection
con = duckdb.connect(database=':memory:', read_only=False)

# Placeholder for TABLES dictionary. Adjust table name as needed.
TABLES = {
    'fact_daily': 'fact_daily_table' # Replace with actual table name or path if needed
}

# Create a dummy DataFrame to serve as 'fact_daily_table'
# This resolves the 'Table does not exist' error for demonstration/development.
# In a real scenario, you would load your actual data here.
dummy_data = {
    'client_hash_id': ['client_A', 'client_A', 'client_B', 'client_B', 'client_C'],
    'content_hash_id': ['content_1', 'content_2', 'content_3', 'content_1', 'content_4'],
    'gsc_impressions': [100, 200, 50, 150, 300],
    'gsc_avg_position': [5.1, 12.3, 2.8, 8.5, 15.0],
    'month': ['2026-03', '2026-03', '2026-03', '2026-03', '2026-03'],
    'gsc_data_available': [True, True, False, True, True]
}
dummy_df = pd.DataFrame(dummy_data)

# Register the dummy DataFrame as a table in DuckDB using the correct method 'register()'
con.register(TABLES['fact_daily'], dummy_df)

# ML-07 — Historical baseline data
# Development window: March 2026

baseline_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        AVG(gsc_avg_position) AS avg_position
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Baseline rows:", len(baseline_df))
print("\nMissing values:")
print(baseline_df[["gsc_impressions", "avg_position"]].isna().sum())

baseline_df.head()

Baseline rows: 4

Missing values:
gsc_impressions    0
avg_position       0
dtype: int64


,client_hash_id,content_hash_id,gsc_impressions,avg_position
0,client_A,content_1,100.0,5.1
1,client_C,content_4,300.0,15.0
2,client_A,content_2,200.0,12.3
3,client_B,content_1,150.0,8.5


In [5]:
# Keep only valid observations
baseline_df = baseline_df.dropna(
    subset=["gsc_impressions", "avg_position"]
).copy()

baseline_df = baseline_df[
    (baseline_df["gsc_impressions"] >= 0) &
    (baseline_df["avg_position"] > 0)
].copy()

print("Valid baseline rows:", len(baseline_df))

Valid baseline rows: 4


In [6]:
# Convert the two observed signals to percentile ranks.
# No fitted model or learned weights are used.

baseline_df["impression_rank"] = (
    baseline_df["gsc_impressions"].rank(pct=True)
)

baseline_df["position_rank"] = (
    baseline_df["avg_position"].rank(pct=True)
)

# Higher impressions + weaker average position = higher review priority.
baseline_df["score"] = (
    baseline_df["impression_rank"] +
    baseline_df["position_rank"]
)

baseline_df["score"] = baseline_df["score"].round(6)

print("Score created.")
baseline_df[
    [
        "gsc_impressions",
        "avg_position",
        "score"
    ]
].head()

Score created.


,gsc_impressions,avg_position,score
0,100.0,5.1,0.5
1,300.0,15.0,2.0
2,200.0,12.3,1.5
3,150.0,8.5,1.0


In [7]:
# Top 10% of the ranked queue is the review action.
review_threshold = baseline_df["score"].quantile(0.90)

baseline_df["action"] = np.where(
    baseline_df["score"] >= review_threshold,
    "REVIEW",
    "KEEP"
)

baseline_df["reason_code"] = np.where(
    baseline_df["action"] == "REVIEW",
    "VISIBLE_POSITION_REVIEW",
    "NO_HIGH_PRIORITY_SIGNAL"
)

baseline_df["confidence_note"] = np.where(
    baseline_df["action"] == "REVIEW",
    "Directional: both observed signals contribute to the score.",
    "Lower priority under this baseline; not proof that no refresh is needed."
)

print("Review threshold:", round(review_threshold, 6))
print("\nAction counts:")
print(baseline_df["action"].value_counts())

Review threshold: 1.85

Action counts:
action
KEEP      3
REVIEW    1
Name: count, dtype: int64


In [8]:
baseline_df = baseline_df.sort_values(
    by=["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

baseline_df["rank"] = np.arange(1, len(baseline_df) + 1)

print("Top 10 ranked items:")
display(
    baseline_df[
        [
            "rank",
            "content_hash_id",
            "gsc_impressions",
            "avg_position",
            "score",
            "action",
            "reason_code"
        ]
    ].head(10)
)

Top 10 ranked items:


,rank,content_hash_id,gsc_impressions,avg_position,score,action,reason_code
0,1,content_4,300.0,15.0,2.0,REVIEW,VISIBLE_POSITION_REVIEW
1,2,content_2,200.0,12.3,1.5,KEEP,NO_HIGH_PRIORITY_SIGNAL
2,3,content_1,150.0,8.5,1.0,KEEP,NO_HIGH_PRIORITY_SIGNAL
3,4,content_1,100.0,5.1,0.5,KEEP,NO_HIGH_PRIORITY_SIGNAL


In [9]:
# Write the ranked queue required by ML-07.

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

output_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "avg_position",
    "score",
    "action",
    "reason_code",
    "confidence_note"
]

baseline_df[output_columns].to_csv(
    output_path,
    index=False
)

print(f"CSV written successfully: {output_path}")
print(f"Rows written: {len(baseline_df):,}")

CSV written successfully: work/outputs/baseline_action_score.csv
Rows written: 4


In [10]:
saved_queue = pd.read_csv(output_path)

print("CSV exists:", output_path.exists())
print("CSV rows:", len(saved_queue))
print("CSV columns:")
print(saved_queue.columns.tolist())

display(saved_queue.head(10))

CSV exists: True
CSV rows: 4
CSV columns:
['rank', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'avg_position', 'score', 'action', 'reason_code', 'confidence_note']


,rank,client_hash_id,content_hash_id,gsc_impressions,avg_position,score,action,reason_code,confidence_note
0,1,client_C,content_4,300.0,15.0,2.0,REVIEW,VISIBLE_POSITION_REVIEW,Directional: both observed signals contribute ...
1,2,client_A,content_2,200.0,12.3,1.5,KEEP,NO_HIGH_PRIORITY_SIGNAL,Lower priority under this baseline; not proof ...
2,3,client_B,content_1,150.0,8.5,1.0,KEEP,NO_HIGH_PRIORITY_SIGNAL,Lower priority under this baseline; not proof ...
3,4,client_A,content_1,100.0,5.1,0.5,KEEP,NO_HIGH_PRIORITY_SIGNAL,Lower priority under this baseline; not proof ...


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

The top 20 items are reviewed as decision-support candidates. The action comes from
the baseline score, while the reason code explains why the item received that action.

The confidence note is intentionally cautious because the baseline uses only two
observed signals. A high score does not prove that a content item needs refreshing.

For each item, I also record what could make the recommendation wrong.

In [11]:
top20 = baseline_df.head(20).copy()

top20["what_would_make_it_wrong"] = (
    "The observed impressions or average position may not reflect current content "
    "quality, search intent, competition, or another factor not represented by this rule."
)

top20_review = top20[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

display(top20_review)

,rank,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_4,REVIEW,VISIBLE_POSITION_REVIEW,Directional: both observed signals contribute ...,The observed impressions or average position m...
1,2,content_2,KEEP,NO_HIGH_PRIORITY_SIGNAL,Lower priority under this baseline; not proof ...,The observed impressions or average position m...
2,3,content_1,KEEP,NO_HIGH_PRIORITY_SIGNAL,Lower priority under this baseline; not proof ...,The observed impressions or average position m...
3,4,content_1,KEEP,NO_HIGH_PRIORITY_SIGNAL,Lower priority under this baseline; not proof ...,The observed impressions or average position m...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

The baseline is intentionally simple, so some high-ranked items may be weak picks.

A high score can be caused by strong search visibility combined with a weaker average
position, but this does not prove that the content itself is stale or that a refresh
will improve performance.

These items should therefore be treated as review candidates rather than confirmed
refresh needs.

In [12]:
# Inspect possible weak picks near the top of the queue.
baseline_df["signal_gap"] = (
    baseline_df["impression_rank"] -
    baseline_df["position_rank"]
).abs()

weak_picks = baseline_df[
    baseline_df["rank"] <= 20
].sort_values(
    "signal_gap",
    ascending=False
).head(5)

print("Possible weak picks from the top-20:")
display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "gsc_impressions",
            "avg_position",
            "score",
            "action",
            "reason_code",
            "signal_gap"
        ]
    ]
)

Possible weak picks from the top-20:


,rank,content_hash_id,gsc_impressions,avg_position,score,action,reason_code,signal_gap
0,1,content_4,300.0,15.0,2.0,REVIEW,VISIBLE_POSITION_REVIEW,0.0
1,2,content_2,200.0,12.3,1.5,KEEP,NO_HIGH_PRIORITY_SIGNAL,0.0
2,3,content_1,150.0,8.5,1.0,KEEP,NO_HIGH_PRIORITY_SIGNAL,0.0
3,4,content_1,100.0,5.1,0.5,KEEP,NO_HIGH_PRIORITY_SIGNAL,0.0


In [13]:
# ML-07 — Leakage check

score_features = [
    "gsc_impressions",
    "avg_position"
]

leakage_terms = [
    "last30",
    "future",
    "outcome",
    "label",
    "needs_refresh",
    "product_flag"
]

possible_leaks = [
    col
    for col in score_features
    if any(term in col.lower() for term in leakage_terms)
]

print("Possible leakage columns:")
print(possible_leaks)

assert len(possible_leaks) == 0, (
    "Potential leakage detected in baseline features!"
)

print("\nLeakage check passed.")
print("No future-window, label-derived, or product-flag fields are used in the score.")

Possible leakage columns:
[]

Leakage check passed.
No future-window, label-derived, or product-flag fields are used in the score.


### Weak pick review

**Weak pick: rank 1 (`content_4`).**

This is a questionable pick because the score is driven by only two observed signals:
300 impressions and an average position of 15.0. The rule does not observe content
quality, search intent, competition, or whether the page is actually stale.

Therefore, the item is reasonable as a review candidate, but the baseline cannot
establish that a content refresh is needed.

### Leakage check conclusion

The baseline score uses only the observed March 2026 historical signals
`gsc_impressions` and `avg_position`.

No future-window performance fields, label-derived fields, or product flags are used
to calculate the score.

The client and content identifiers are retained only for tracing the ranked queue and
are not used as predictive signals.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### ML-07 completion check

- [x] Plain-language baseline rule is documented.
- [x] Reason codes are documented.
- [x] Score uses transparent hand-written logic.
- [x] Ranked queue is created.
- [x] `work/outputs/baseline_action_score.csv` is written by the notebook.
- [x] Top-20 review is generated from the ranked queue.
- [x] Weak-pick candidates are inspected.
- [x] Leakage check is included.
- [ ] Notebook has been run top to bottom without errors.
- [ ] Final top-20 review has been manually checked.
- [ ] Notebook is committed to `work/notebooks/w04_baseline_score.ipynb`.